# SSD 配對交易滾動回測系統：完整交易邏輯分析

---

## 目錄

1. [系統架構總覽](#1-系統架構總覽)
2. [Phase 1：資料前處理（DataProcessor）](#2-phase-1資料前處理dataprocessor)
3. [Phase 2：形成期（Formation）](#3-phase-2形成期formation)
4. [Phase 3：交易期（Trading）](#4-phase-3交易期trading)
5. [Phase 4：滾動回測引擎（RollingBacktester）](#5-phase-4滾動回測引擎rollingbacktester)
6. [Phase 5：資金管理與結果匯出](#6-phase-5資金管理與結果匯出)
7. [完整公式彙整表](#7-完整公式彙整表)
8. [參數敏感度與設計決策說明](#8-參數敏感度與設計決策說明)

---

## 1. 系統架構總覽

本系統為「滾動視窗配對交易（Rolling-Window Pairs Trading）」框架，核心思想是：**在形成期（Formation Period）找出價格走勢最相近的股票對，並在交易期（Trading Period）透過Z-Score訊號進行均值回歸套利**。

```
┌─────────────────────────────────────────────────────┐
│                   RollingBacktester                  │
│   (滾動視窗排程 + 網格搜尋 Grid Search)               │
│                                                     │
│  ┌──────────────┐      ┌──────────────────────────┐ │
│  │ DataProcessor│ ───> │  Formation（形成期模組）  │ │
│  │ 資料前處理   │      │  - SSD 最小配對篩選       │ │
│  └──────────────┘      │  - ADF 共整合檢驗         │ │
│                        │  - OU 半衰期過濾           │ │
│                        │  - Hurst 指數篩選          │ │
│                        └────────────┬─────────────┘ │
│                                     │ selected_pairs │
│                        ┌────────────▼─────────────┐ │
│                        │  Trading（交易期模組）    │ │
│                        │  - Z-Score 計算           │ │
│                        │  - 進出場邏輯              │ │
│                        │  - 停損機制               │ │
│                        │  - PnL 追蹤               │ │
│                        └──────────────────────────┘ │
└─────────────────────────────────────────────────────┘
```

**滾動視窗時間軸示意（formation=252, trading=126, step=21）：**

```
時間軸 ──────────────────────────────────────────────────────────────────►

期1: [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
期2:       [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
期3:             [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
...
期6:                               [════════════ 形成期 252天 ════════════│════════ 交易期 126天 ════════]
                 ◄─ step=21 ─►

※ 每期僅錯開 rolling_step=21 天，交易期大量重疊並行（最多同時 6 期）
※ 並行期數上限：max_concurrent = trading_window // rolling_step = 126 // 21 = 6
※ 資金切為 6 槽輪流承接，每槽在前一期交易期結束後才釋放供下一期使用
```

---

## 2. Phase 1：資料前處理（DataProcessor）

### 2.1 資料載入

從 SQLite 資料庫讀取歷史收盤價，優先使用還原股價（`Adj_Close`），若無則使用原始收盤價（`Close`）：

```sql
SELECT Date, Symbol, COALESCE(Adj_Close, Close) AS price
FROM daily_prices
WHERE COALESCE(Adj_Close, Close) IS NOT NULL
ORDER BY Date ASC
```

### 2.2 資料清洗四步驟

原始資料轉為寬格式 Pivot 後，依序執行以下清洗：

**Step A：移除高缺失值標的**

$$\text{保留條件：} \frac{\text{NaN 筆數}}{\text{總交易日數}} < 0.20$$

**Step B：向前填補（Forward Fill）**

$$P_t = P_{t-k}, \quad k = \min(\text{連續缺失天數},\ 5)$$

僅允許最多向前填補 5 個交易日，避免長期缺口造成失真。

**Step C：移除高缺失值日期（列）**

$$\text{保留條件：} \frac{\text{該日缺失標的數}}{\text{總標的數}} \leq 0.10$$

**Step D：移除覆蓋率不足標的**

$$\text{保留條件：非 NaN 筆數} \geq \lfloor N_{total} \times 0.90 \rfloor$$

### 2.3 滾動視窗切割

- **回測起點**（`backtest_start`）：決定第一個可交易日期
- **延伸起點**（`data_slice_start`）：往前延伸 `formation_window` 天，確保第一期形成期有足夠資料

$$\text{data\_slice\_start} = \text{all\_dates}[\max(0,\ \text{first\_trade\_idx} - \text{formation\_window})]$$

---

## 3. Phase 2：形成期（Formation）

形成期的目標是：**在同一產業內，找出歷史價格走勢最相似的股票對**，並通過統計檢驗確保其具備均值回歸特性。

### 3.1 對數價格正規化（Z-Score Normalization）

**Step 1：取自然對數**

$$\ell_{i,t} = \ln(P_{i,t})$$

**Step 2：計算形成期均值與標準差**

$$\bar{\ell}_i = \frac{1}{T}\sum_{t=1}^{T}\ell_{i,t}, \quad \sigma_{\ell_i} = \sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(\ell_{i,t} - \bar{\ell}_i)^2}$$

**Step 3：Z-Score 正規化**

$$\tilde{P}_{i,t} = \frac{\ell_{i,t} - \bar{\ell}_i}{\sigma_{\ell_i}}$$

此正規化目的：消除不同標的間的價格量級差異，使 SSD 比較具有意義。

### 3.2 SSD 計算（Sum of Squared Differences）

對於任意配對 $(A, B)$，計算正規化對數價格序列的差異平方和：

$$\text{SSD}(A, B) = \sum_{t=1}^{T}\left(\tilde{P}_{A,t} - \tilde{P}_{B,t}\right)^2$$

等價於 Euclidean 距離的平方：

$$\text{SSD}(A, B) = \|\tilde{\mathbf{P}}_A - \tilde{\mathbf{P}}_B\|^2$$

**實作最佳化：使用 `scipy.spatial.distance.pdist` 搭配 `metric='sqeuclidean'`**，一次性向量化計算所有配對，避免巢狀迴圈。

### 3.3 避險比率（Hedge Ratio）計算

對於配對 $(A, B)$，以 $B$ 為自變數（X），$A$ 為因變數（Y），利用協方差矩陣估計 $\beta$（OLS 斜率）：

$$\hat{\beta} = \frac{\text{Cov}(\tilde{P}_B, \tilde{P}_A)}{\text{Var}(\tilde{P}_B)}$$

### 3.4 價差（Spread）計算

$$S_t = \tilde{P}_{A,t} - \hat{\beta} \cdot \tilde{P}_{B,t}$$

### 3.5 統計篩選三關卡

SSD 最小的前 $\min(200,\ \text{top\_n} \times 15)$ 組候選配對，依序通過以下三個檢驗：

---

#### 關卡 A：ADF 共整合檢驗（Augmented Dickey-Fuller Test）

檢驗虛無假設 $H_0$：價差序列 $S_t$ 存在單位根（隨機漫步）。

ADF 迴歸（無截距）：

$$\Delta S_t = \rho S_{t-1} + \sum_{k=1}^{p} \gamma_k \Delta S_{t-k} + \varepsilon_t$$

**篩選條件：** $p\text{-value} < 0.05$（拒絕單位根，確認平穩性）

---

#### 關卡 B：Ornstein-Uhlenbeck 半衰期（Half-Life）

OU 過程離散化近似（OLS 迴歸）：

$$\Delta S_t = \alpha + \lambda S_{t-1} + \varepsilon_t$$

其中 $\lambda$ 為均值回歸速度（應為負值）。

**半衰期公式：**

$$\tau_{1/2} = -\frac{\ln 2}{\lambda}$$

**篩選條件：**

$$\lambda < 0 \quad \text{且} \quad 2.0 \leq \tau_{1/2} \leq 40.0 \text{（交易日）}$$

- $\lambda \geq 0$：價差發散，不具均值回歸性
- $\tau_{1/2} < 2$：回歸太快，難以捕捉（訊號噪音大）
- $\tau_{1/2} > 40$：回歸太慢，資金占用過長

---

#### 關卡 C：Hurst 指數篩選（R/S Analysis）

以 R/S 分析近似 Hurst 指數，衡量時間序列的長記憶性：

**對各分段長度 $n$ 計算 R/S 統計量：**

$$R/S(n) = \frac{\max_{1 \leq k \leq n}\sum_{i=1}^{k}(X_i - \bar{X}) - \min_{1 \leq k \leq n}\sum_{i=1}^{k}(X_i - \bar{X})}{\sigma_n}$$

其中 $X_i = \Delta S_i$（一階差分）。

**以 OLS 迴歸估計 Hurst 指數：**

$$\ln(R/S) \approx H \cdot \ln(n) + C$$

**Hurst 指數解讀：**

| $H$ 範圍     | 意義               |
|--------------|--------------------|
| $H < 0.5$    | 均值回歸（Anti-persistent） |
| $H = 0.5$    | 隨機漫步            |
| $H > 0.5$    | 趨勢追蹤（Persistent） |

**篩選條件：** $H < 0.40$（強均值回歸特性）

---

### 3.6 形成期輸出

通過三關卡篩選後，記錄以下統計量供交易期使用：

$$\mu_S = \frac{1}{T}\sum_{t=1}^{T}S_t, \quad \sigma_S = \sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(S_t - \mu_S)^2}$$

輸出欄位：`Form_Start`, `Form_End`, `Sector`, `Ticker_A`, `Ticker_B`, `SSD`, `Hedge_Ratio` ($\hat{\beta}$), `Spread_Mean` ($\mu_S$), `Spread_Std` ($\sigma_S$), `Log_Mean_A/B`, `Log_Std_A/B`

---

## 4. Phase 3：交易期（Trading）

### 4.1 Z-Score 計算模式

交易期的核心是將每日價差轉換為 Z-Score，作為進出場的標準化訊號。

系統支援兩種模式：

---

#### 模式一：固定參數（`zscore_window = 0`）

使用形成期計算的均值 $\mu_S$ 與標準差 $\sigma_S$：

**正規化（使用形成期參數）：**

$$\tilde{P}_{A,t} = \frac{\ln P_{A,t} - \bar{\ell}_A}{\sigma_{\ell_A}}, \quad \tilde{P}_{B,t} = \frac{\ln P_{B,t} - \bar{\ell}_B}{\sigma_{\ell_B}}$$

**價差：**

$$S_t = \tilde{P}_{A,t} - \hat{\beta} \cdot \tilde{P}_{B,t}$$

**Z-Score（固定參數）：**

$$Z_t = \text{clip}\left(\frac{S_t - \mu_S}{\sigma_S},\ -Z_{clip},\ +Z_{clip}\right)$$

---

#### 模式二：滾動視窗（`zscore_window = W > 0`）

使用滾動視窗動態估計 $\beta$、均值與標準差：

**滾動 $\beta$（Kalman-like OLS）：**

$$\hat{\beta}_t^{roll} = \frac{\text{Cov}_W(\tilde{P}_B, \tilde{P}_A)}{\text{Var}_W(\tilde{P}_B)}$$

**滾動截距：**

$$\hat{\alpha}_t^{roll} = \bar{\tilde{P}}_{A,W} - \hat{\beta}_t^{roll} \cdot \bar{\tilde{P}}_{B,W}$$

**滾動殘差（價差）：**

$$S_t^{roll} = \tilde{P}_{A,t} - \hat{\alpha}_t^{roll} - \hat{\beta}_t^{roll} \cdot \tilde{P}_{B,t}$$

**滾動殘差標準差（利用方差分解）：**

$$\sigma_{S,t}^{roll} = \sqrt{\max\left(\text{Var}_W(\tilde{P}_A) - \hat{\beta}_t^{roll} \cdot \text{Cov}_W(\tilde{P}_B, \tilde{P}_A),\ 0\right)}$$

**Z-Score（滾動模式）：**

$$Z_t = \text{clip}\left(\frac{S_t^{roll}}{\max(\sigma_{S,t}^{roll},\ \sigma_{min})},\ -Z_{clip},\ +Z_{clip}\right)$$

---

#### 選配：波動度調整（`use_vol_adjust = True`）

計算近 20 日的滾動標準差，相對於形成期標準差的比值，調整分母以適應波動度擴張：

$$\text{vol\_factor}_t = \max\left(1.0,\ \frac{\sigma_{S,20d}}{\sigma_S^{form}}\right)$$

$$\sigma_{adj,t} = \max(\sigma_t \cdot \text{vol\_factor}_t,\ \sigma_{min})$$

### 4.2 資金分配（Position Sizing）

對於每個配對，總資金為 `capital_per_pair`（$C$）：

**按避險比率比例分配：**

$$V_A = \frac{C}{1 + |\hat{\beta}|}, \quad V_B = \frac{C \cdot |\hat{\beta}|}{1 + |\hat{\beta}|}$$

其中 $V_A$ 為 Ticker_A 的名義部位價值，$V_B$ 為 Ticker_B 的名義部位價值，滿足：

$$V_A + V_B = C$$

**股數計算：**

$$N_A = \frac{V_A}{P_A}, \quad N_B = \frac{V_B}{P_B}$$

### 4.3 進場邏輯（Entry）

**Short Spread（做空價差，$Z_t > Z_{entry}$）：**

意義：$A$ 相對 $B$ 被高估，預期價差縮小。

$$\text{Position} = -1: \quad N_A^{pos} = -\frac{V_A}{P_A}\ (\text{放空 A}),\quad N_B^{pos} = +\frac{V_B}{P_B}\ (\text{做多 B})$$

**Long Spread（做多價差，$Z_t < -Z_{entry}$）：**

意義：$A$ 相對 $B$ 被低估，預期價差擴張後回歸。

$$\text{Position} = +1: \quad N_A^{pos} = +\frac{V_A}{P_A}\ (\text{做多 A}),\quad N_B^{pos} = -\frac{V_B}{P_B}\ (\text{放空 B})$$

**冷卻方向機制（Cooldown Direction）：**

平倉後，系統記錄方向 `cooldown_dir`，防止在均值尚未回復前立即反向建倉：

- 若 `cooldown_dir = -1`（上次為 Short），則等 $Z_t \leq Z_{exit}$ 後才解除限制
- 若 `cooldown_dir = +1`（上次為 Long），則等 $Z_t \geq -Z_{exit}$ 後才解除限制

### 4.4 未實現損益（Unrealized PnL）

$$\text{UnrPnL}_t = N_A^{pos}(P_{A,t} - P_A^{entry}) + N_B^{pos}(P_{B,t} - P_B^{entry}) - F_{entry} - F_{exit}^{est}$$

其中交易成本（手續費 + 滑價）：

$$F_{entry} = (|N_A^{pos}| \cdot P_A^{entry} + |N_B^{pos}| \cdot P_B^{entry}) \cdot r_{friction}$$

$$F_{exit}^{est} = (|N_A^{pos}| \cdot P_{A,t} + |N_B^{pos}| \cdot P_{B,t}) \cdot r_{friction}$$

$$r_{friction} = r_{fee} + r_{slippage}$$

### 4.5 平倉邏輯（Exit）

**正常平倉條件：**

- Short Spread 平倉：$Z_t \leq Z_{exit}$（價差縮回至正常範圍）
- Long Spread 平倉：$Z_t \geq -Z_{exit}$（價差縮回至正常範圍）

**平倉已實現損益：**

$$\text{ClosedPnL} = N_A^{pos}(P_{A,t}^{exit} - P_A^{entry}) + N_B^{pos}(P_{B,t}^{exit} - P_B^{entry}) - F_{entry} - F_{exit}^{actual}$$

**累積損益更新：**

$$\text{RealizedPnL} \mathrel{+}= \text{ClosedPnL}$$

### 4.6 停損機制（Stop Loss）

系統支援三層停損：

---

#### 層級 1：個別配對資金停損（`stop_loss_pct`）

$$\text{觸發條件：} \frac{-\text{UnrPnL}_t}{C} \geq \text{stop\_loss\_pct}$$

例如 `stop_loss_pct = 0.05` 代表虧損超過該配對資金 5% 時強制平倉。

---

#### 層級 2：動態 Z-Score 停損（`use_dynamic_stop`, `dynamic_stop_z`）

$$\text{觸發條件：} |Z_t| > Z_{dyn\_stop}$$

當 Z-Score 繼續向不利方向擴張超過動態停損閾值，強制平倉防止進一步虧損。

---

#### 層級 3：投資組合總體停損（`portfolio_stop_loss_pct`）

**後置斷路器（Portfolio Circuit Breaker）：**

計算所有配對的每日累積損益總和：

$$\text{Portfolio\_PnL}_t = \sum_{i=1}^{N} \text{CumPnL}_{i,t}$$

$$\text{總資金：} C_{total} = C \times N_{pairs}$$

$$\text{觸發條件：} \frac{\text{Portfolio\_PnL}_t}{C_{total}} \leq -\text{portfolio\_stop\_loss\_pct}$$

**觸發後處理：**

在觸發日（`cutoff_date`）之後的所有交易日，所有配對強制清倉並凍結損益：

$$\forall t > t_{cutoff}: \quad \text{Position} = 0,\ \text{Unrealized} = 0,\ \text{Status} = \text{STOPPED}$$

### 4.7 期末強制平倉（Period End Exit）

若交易期結束時仍有未平倉部位，以最後一日收盤價強制平倉：

$$\text{FinalClosedPnL} = N_A^{pos}(P_{A,T} - P_A^{entry}) + N_B^{pos}(P_{B,T} - P_B^{entry}) - F_{entry} - F_{exit}$$

### 4.8 每日損益變化（Daily Delta）

$$\Delta_t = \text{CumPnL}_t - \text{CumPnL}_{t-1}$$

此欄位用於後續計算期間總損益（對所有配對加總後積分）。

---

## 5. Phase 4：滾動回測引擎（RollingBacktester）

### 5.1 滾動視窗排程

設定：
- `formation_window` = $W_F$（形成期長度，例如 252 天）
- `trading_window` = $W_T$（交易期長度，例如 126 天）
- `rolling_step` = $\Delta$（每次滾動步長，例如 21 天）

第 $k$ 期的日期索引：

$$\text{形成期：} [t_k - W_F,\ t_k)$$
$$\text{交易期：} [t_k,\ t_k + W_T)$$
$$t_{k+1} = t_k + \Delta$$

**關鍵特性：交易期大量重疊並行**

由於 $\Delta \ll W_T$（步進量遠小於交易期長度），相鄰兩期的交易期會大幅重疊：

$$\text{重疊天數} = W_T - \Delta = 126 - 21 = 105 \text{ 天}$$

因此任意時刻最多有以下數量的交易期同時進行：

$$N_{slots} = \left\lfloor \frac{W_T}{\Delta} \right\rfloor = \left\lfloor \frac{126}{21} \right\rfloor = 6$$

**資金槽（Slot）機制：**

初始資金 $C_{total}$ 被切分為 $N_{slots}$ 份，每份獨立追蹤複利：

$$C_{slot} = \frac{C_{total}}{N_{slots}}$$

當某期交易結束（`trade_end_idx` 到達），該槽釋放，分配給下一個等待的交易期。若所有槽均忙碌，則選擇最早結束的槽（最小的 `avail_idx`）優先覆蓋。

### 5.2 網格搜尋（Grid Search）

對以下參數的笛卡兒積（Cartesian Product）進行全面搜尋：

| 參數 | 說明 |
|------|------|
| `top_n` | 選取最佳配對數 |
| `stop_loss_pct` | 個別配對停損比率 |
| `zscore_window` | Z-Score 滾動視窗（0 = 固定） |
| `portfolio_stop_loss_pct` | 投資組合總體停損 |
| `max_sector_ratio` | 單一產業配對上限比率 |
| `dynamic_stop_z` | 動態 Z-Score 停損閾值 |
| `use_vol_adjust` | 是否啟用波動度調整 |

**總搜尋組合數：**

$$N_{combos} = \prod_{p \in \text{params}} |\text{p\_list}|$$

### 5.3 資本更新（Capital Compounding）

每個參數組合維護獨立的資金槽陣列。每期結束後更新資金：

$$C_{slot}^{k+1} = \max\left(0,\ C_{slot}^{k} + \text{PeriodPnL}^k\right)$$

**期間損益計算：**

$$\text{PeriodPnL}^k = \sum_{t \in \text{交易期}} \sum_{i=1}^{N} \Delta_{i,t}$$

### 5.4 產業分散化過濾（`max_sector_ratio`）

當 `sec_ratio > 0` 時，每個產業的入選配對數上限：

$$\text{max\_per\_sector} = \max\left(1,\ \lfloor \text{top\_n} \times \text{sec\_ratio} \rfloor\right)$$

例如 `top_n = 20`, `sec_ratio = 0.3`，則每個產業最多選入 6 組配對。

---

## 6. Phase 5：資金管理與結果匯出

### 6.1 每配對資金分配

$$C_{pair} = \frac{C_{slot}}{N_{pairs}}$$

其中 $N_{pairs}$ = `top_n`（選定的配對數）。

### 6.2 期間損益彙總

每期結束後，對該期所有配對所有日期的 `Daily_Delta` 加總：

$$\text{PeriodPnL} = \sum_{i}^{N_{pairs}} \sum_{t}^{T_{trade}} \Delta_{i,t}$$

### 6.3 結果輸出至 SQLite

每種參數組合產生一份完整交易紀錄 DataFrame，包含以下欄位：

| 欄位 | 說明 |
|------|------|
| `Date` | 交易日期 |
| `Price_A / Price_B` | 當日收盤價 |
| `Hedge_Ratio` | 當日避險比率 $\hat{\beta}$ |
| `ZScore` | 當日 Z-Score $Z_t$ |
| `Position` | 持倉方向（+1 / -1 / 0） |
| `Unrealized_PnL` | 未實現損益 |
| `Realized_PnL` | 累計已實現損益 |
| `Cumulative_PnL` | 總損益（含未實現） |
| `Status` | 當日狀態（見下表） |
| `Trade_PnL` | 本次交易損益（平倉時） |
| `Days_Held` | 持倉天數 |
| `Daily_Delta` | 每日損益變化 $\Delta_t$ |

**Status 狀態碼說明：**

| Status | 說明 |
|--------|------|
| `HOLD_CASH` | 無持倉，等待訊號 |
| `HOLD_CASH (COOLDOWN)` | 冷卻中，禁止進場 |
| `ENTER_LONG_A` | 做多 A / 放空 B |
| `ENTER_SHORT_A` | 放空 A / 做多 B |
| `HOLDING` | 持倉中 |
| `EXIT` | 正常平倉 |
| `STOP_LOSS_TRIGGERED` | 個別停損觸發 |
| `PERIOD_END_EXIT` | 期末強制平倉 |
| `PORTFOLIO_STOP_TRIGGERED` | 組合停損觸發 |
| `STOPPED` | 已永久停止交易 |

---

## 7. 完整公式彙整表

### 形成期公式

| 名稱 | 公式 |
|------|------|
| 對數價格 | $\ell_{i,t} = \ln P_{i,t}$ |
| Z-Score 正規化 | $\tilde{P}_{i,t} = (\ell_{i,t} - \bar{\ell}_i) / \sigma_{\ell_i}$ |
| SSD | $\text{SSD}(A,B) = \sum_t (\tilde{P}_{A,t} - \tilde{P}_{B,t})^2$ |
| 避險比率（OLS） | $\hat{\beta} = \text{Cov}(\tilde{P}_B, \tilde{P}_A) / \text{Var}(\tilde{P}_B)$ |
| 價差 | $S_t = \tilde{P}_{A,t} - \hat{\beta} \cdot \tilde{P}_{B,t}$ |
| ADF 迴歸 | $\Delta S_t = \rho S_{t-1} + \sum_k \gamma_k \Delta S_{t-k} + \varepsilon_t$ |
| OU 迴歸 | $\Delta S_t = \alpha + \lambda S_{t-1} + \varepsilon_t$ |
| 半衰期 | $\tau_{1/2} = -\ln 2 / \lambda$ |
| R/S 統計量 | $R/S(n) = (\max - \min \text{ 累積偏差}) / \sigma_n$ |
| Hurst 指數 | $H = d(\ln R/S) / d(\ln n)$ |

### 交易期公式

| 名稱 | 公式 |
|------|------|
| 資金分配（A） | $V_A = C / (1 + |\hat{\beta}|)$ |
| 資金分配（B） | $V_B = C \cdot |\hat{\beta}| / (1 + |\hat{\beta}|)$ |
| 股數 | $N_i = V_i / P_i$ |
| Z-Score（固定） | $Z_t = \text{clip}((S_t - \mu_S)/\sigma_S, -Z_{clip}, +Z_{clip})$ |
| Z-Score（滾動） | $Z_t = \text{clip}(S_t^{roll}/\sigma_{S,t}^{roll}, -Z_{clip}, +Z_{clip})$ |
| 未實現損益 | $\text{UnrPnL}_t = \sum_i N_i(P_{i,t} - P_i^{entry}) - F_{entry} - F_{exit}^{est}$ |
| 交易成本 | $F = (|N_A|P_A + |N_B|P_B) \cdot (r_{fee} + r_{slippage})$ |
| 個別停損條件 | $-\text{UnrPnL}_t / C \geq \text{SL\_pct}$ |
| 組合停損條件 | $\sum_i \text{CumPnL}_{i,t} / C_{total} \leq -\text{PSL\_pct}$ |
| 每日 Delta | $\Delta_t = \text{CumPnL}_t - \text{CumPnL}_{t-1}$ |
| 期間損益 | $\text{PeriodPnL} = \sum_{i,t} \Delta_{i,t}$ |
| 資金複利更新 | $C_{next} = \max(0, C_{curr} + \text{PeriodPnL})$ |

---

## 8. 參數敏感度與設計決策說明

### 8.1 SSD vs. 相關係數

SSD 相較於相關係數的優勢在於：**它同時捕捉走勢方向的相似性與水平位置的接近程度**。兩個高度相關但均值差距大的序列，SSD 會較大，反映其配對交易風險更高。

### 8.2「先初篩再過濾」優化

ADF、OU、Hurst 計算均為 $O(T)$ 的統計迴歸，對大量配對組合（如 S&P 500 約 $\binom{500}{2} \approx 125,000$ 組）直接全量計算耗時極長。

本系統採用兩階段策略：

1. **快速 SSD 排序**（$O(N^2 T)$，向量化，毫秒級）
2. **僅對前 $\max(200, \text{top\_n} \times 15)$ 組候選進行慢速統計檢驗**

**加速效果：** 原本需要 30,000+ 次統計迴歸，降至約 200~300 次，速度提升 **30~50 倍**。

### 8.3 Extended Trade Data 設計

當使用滾動 Z-Score 模式（`zscore_window = W`）時，交易期第一天的 Z-Score 需要前 $W$ 天的歷史資料才能計算。因此系統在讀取交易期資料時，往前延伸 $\max(\text{zscore\_window\_list})$ 天：

$$\text{extended\_start\_idx} = \max(0, \text{trade\_start\_idx} - \max(W_{list}))$$

延伸資料僅用於計算 Z-Score，不產生交易紀錄。

### 8.4 `allow_reentry` 設計差異

| 設定 | 停損後行為 |
|------|------------|
| `allow_reentry = False` | 永久停止該配對交易（`is_stopped = True`） |
| `allow_reentry = True` | 進入冷卻模式（`cooldown_dir` 記錄方向），等 Z-Score 回歸後可再進場 |

### 8.5 Z-Score Clip 的必要性

在極端市場事件（黑天鵝）中，Z-Score 可能出現極端值（如 $|Z| > 20$），若不加限制可能觸發大量誤判。設定 `zscore_clip = 10.0` 可防止：

- 單一異常日期觸發錯誤進場訊號
- 動態 Z-Score 計算中的數值不穩定

---

*文件生成時間：系統自動分析*  
*分析來源：SSD 配對交易滾動回測系統原始碼*